# 03. Первичный пространственный анализ

**ЛР 2, шаг 8.** Цель: получить первые ответы на вопросы заказчика и проверить, что выбранные данные и методы работают. Это ещё не продукт, а проверка гипотез, на основе которой уточняются бэклог и план.

Пример AquaMap отвечает на три вопроса:
1. Сколько точек питьевой воды в каждом районе и какова их плотность?
2. Какая доля жилых зданий находится в пределах 500 м от ближайшей точки?
3. Какие парки остаются без питьевой воды?

Замените вопросы и операции своими. Типовые операции: проекция в метры, буфер, пространственное объединение (spatial join), ближайший сосед, пересечение (overlay), агрегирование по районам, картограмма (choropleth).\n\n[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OWNER/REPO/blob/main/notebooks/03_spatial_analysis.ipynb)\n\nЧтобы кнопка открывала ваш ноутбук, замените в ссылке `OWNER/REPO` на владельца и имя своего репозитория (например, `ivan-p/pm-lab-00-aquamap`). Другие способы открыть ноутбук описаны в приложении Б.

## Настройка среды

Ноутбук рассчитан на **Google Colab** (как получить бесплатный доступ: приложение Б к заданию ЛР 2). Он работает и локально, в Jupyter или VS Code.

Что делает ячейка ниже:
1. В Colab ставит недостающие библиотеки и подключает Google Диск. Colab попросит разрешение на доступ к Диску: выберите тот же аккаунт Google, под которым открыт Colab.
2. Находит рабочую папку группы на Диске. Все данные и результаты сохраняются туда, поэтому после отключения сессии ничего не пропадёт.
3. Создаёт внутри неё ту же структуру папок, что и в репозитории: `data/raw`, `data/manual`, `data/processed`, `docs/img`, `docs/maps`.

**Перед первым запуском** впишите в `PROJECT_FOLDER` имя папки группы на Google Диске.

Запуск всего ноутбука: **Среда выполнения → Выполнить все** (Runtime → Run all).

In [ ]:
# === Настройка среды: запустите эту ячейку первой ===
import sys, subprocess
from pathlib import Path

# Имя папки группы в «Моём диске» Google. Обычно совпадает с именем репозитория.
PROJECT_FOLDER = "pm-lab-5-CIA"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'geopandas>=1.0', 'mapclassify', 'folium'], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / PROJECT_FOLDER
    if not BASE.exists():
        raise FileNotFoundError(
            f"На Google Диске нет папки «{PROJECT_FOLDER}». Создайте её или добавьте ярлык "
            "общей папки группы в «Мой диск» (приложение Б, шаг 2) и запустите ячейку снова.")
else:
    BASE = Path("..")   # локальный запуск из папки notebooks/ репозитория

RAW, MANUAL, PROCESSED = BASE / "data" / "raw", BASE / "data" / "manual", BASE / "data" / "processed"
IMG, MAPS = BASE / "docs" / "img", BASE / "docs" / "maps"
for folder in (RAW, MANUAL, PROCESSED, IMG, MAPS):
    folder.mkdir(parents=True, exist_ok=True)
print("Среда:", "Google Colab" if IN_COLAB else "локальная", "· рабочая папка:", BASE.resolve())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

boundary  = gpd.read_file(PROCESSED / "boundary.gpkg")
METRIC_CRS = boundary.estimate_utm_crs()

districts = gpd.read_file(PROCESSED / "districts.gpkg").to_crs(METRIC_CRS)[["name", "geometry"]]
roads     = gpd.read_file(PROCESSED / "roads.gpkg").to_crs(METRIC_CRS)

# Загружаем ДТП (файл лежит в data/raw)
acc = pd.read_csv(RAW / "accidents.csv", encoding="utf-8-sig")
acc["lat"] = pd.to_numeric(acc["lat"], errors="coerce")
acc["lon"] = pd.to_numeric(acc["lon"], errors="coerce")
acc_xy = acc.dropna(subset=["lat", "lon"]).copy()

accidents = gpd.GeoDataFrame(acc_xy, geometry=gpd.points_from_xy(acc_xy["lon"], acc_xy["lat"]), crs=4326).to_crs(METRIC_CRS)
print(f"Загружено: {len(districts)} районов, {len(roads)} дорог, {len(accidents)} ДТП")

## Вопрос 1. Где происходят ДТП (агрегация по районам)

In [ ]:
districts["area_km2"] = districts.area / 1e6
zones = districts[["name", "geometry"]] # НЕ переименовываем name, чтобы не было путаницы

# Удаляем старую колонку district из ДТП (из ГИБДД), чтобы избежать конфликта имен
accidents_clean = accidents.drop(columns=["district"], errors="ignore")

# Пространственное соединение: определяем район для каждого ДТП
acc_in = gpd.sjoin(accidents_clean, zones, predicate="within", how="inner")

# Считаем количество ДТП и плотность
districts["accidents"] = districts["name"].map(acc_in.groupby("name").size()).fillna(0).astype(int)
districts["accidents_per_10km2"] = (districts["accidents"] / districts["area_km2"] * 10).round(2)

# Считаем погибших и раненых
acc_in["killed"] = pd.to_numeric(acc_in["killed"], errors="coerce")
acc_in["injured"] = pd.to_numeric(acc_in["injured"], errors="coerce")
severity = acc_in.groupby("name").agg(killed=("killed", "sum"), injured=("injured", "sum"))
districts = districts.merge(severity, left_on="name", right_index=True, how="left").fillna(0)

districts.drop(columns="geometry").sort_values("accidents", ascending=False)

## Вопрос 2. На каких типах дорог происходят ДТП

In [ ]:
# В данных ГИБДД есть колонка road_class (класс дороги). Посмотрим статистику.
acc_in["road_class"] = acc_in["road_class"].replace("", "Не указан")
road_stats = acc_in.groupby("road_class").agg(
    accidents=("kart_id", "count"),
    killed=("killed", "sum"),
    injured=("injured", "sum")
).sort_values("accidents", ascending=False)

road_stats["accident_share_pct"] = (road_stats["accidents"] / len(acc_in) * 100).round(1)
road_stats

## Карты

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
districts.plot(ax=ax, column="accidents_per_10km2", cmap="Reds", legend=True, edgecolor="white",
               legend_kwds={"label": "ДТП на 10 кв. км", "shrink": 0.6})
roads.plot(ax=ax, color="gray", linewidth=0.5, alpha=0.5)
accidents.plot(ax=ax, color="black", markersize=1, alpha=0.3)

for _, r in districts.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r["name"].replace(" район", ""), (p.x, p.y), ha="center", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none", alpha=0.7))

ax.set_title("Плотность ДТП по районам Уфы (© участники OpenStreetMap, ГИБДД)")
ax.set_axis_off()
fig.savefig(IMG / "accidents_by_district.png", dpi=150, bbox_inches="tight")

In [ ]:
m = districts.to_crs(4326).explore(column="accidents_per_10km2", cmap="Reds", legend=True,
                                   tooltip=["name", "accidents", "killed", "injured", "accidents_per_10km2"],
                                   name="Районы (плотность ДТП)")
accidents.to_crs(4326).explore(m=m, color="black", marker_kwds={"radius": 2}, name="ДТП", tooltip=["date", "dtp_type", "road_class"])
import folium; folium.LayerControl().add_to(m)
m.save(MAPS / "analysis_v0.html")
m

## Выводы

1. **Где происходят ДТП?** Больше всего ДТП зафиксировано в Кировском районе (683 шт.), наименьшее — в Дёмском (145 шт.). По плотности на 10 кв. км с огромным отрывом лидирует Советский район (291.43 ДТП/10км²), так как он самый маленький по площади, но очень насыщенный. Самое большое число погибших (22) и раненых (856) также приходится на Кировский район.

2. **На каких дорогах?** Подавляющая часть аварий — 97.5% (2896 ДТП) — происходит на дорогах местного значения. На федеральных трассах происходит 2.4% ДТП (72 шт.), на региональных — всего 0.1%. Это логично: дороги местного значения составляют основу уличной сети города, где и происходит основной трафик.

3. **Ограничения:** В данных ГИБДД только ДТП с пострадавшими, что занижает общее число аварий (мелкие аварии без пострадавших не учитываются). В OSM загружены только 6 основных классов дорог, поэтому 8.5% ДТП (из отчета о качестве) могут не привязываться к дорогам. Погодные данные (DS-03) пока не загружены, поэтому влияние погоды не оценено.

4. **Для продукта:** Аварийность крайне неравномерна по районам. Гипотеза проекта подтверждается: заказчику нужна карта плотности ДТП, чтобы точечно направлять инспекторов ГИБДД или ремонтные бригады в самые проблемные зоны (например, в Советский и Кировский районы).

5. **Риски:** R1 (неполнота данных OSM) подтвержден частично: для дорог местного значения (где 97.5% аварий) данных может не хватать, но общая картина по районам строится корректно. Требуется дополнительная проверка координат на дорогах местного значения.

## Перенос результатов в репозиторий

Карты из `docs/img` и `docs/maps` должны попасть в репозиторий через Pull Request. В Colab ячейка ниже соберёт их в архив `results_03.zip` и скачает на компьютер. Распакуйте архив и загрузите файлы в те же папки репозитория в своей ветке: **Add file → Upload files** (подробно в приложении Б, шаг 6).

Файлы больше 10 МБ в репозиторий не загружайте: ячейка о них предупредит. При локальной работе результаты уже лежат в папках репозитория, ячейку можно пропустить.

In [ ]:
import shutil
FOLDERS = [IMG, MAPS]
if IN_COLAB:
    from google.colab import files
    pack = Path("/content/results_03")
    shutil.rmtree(pack, ignore_errors=True)
    for folder in FOLDERS:
        for f in folder.rglob("*"):
            if f.is_file() and f.name != ".gitkeep":
                if f.stat().st_size > 10 * 1024 * 1024:
                    print(f"Пропущен (больше 10 МБ): {f.relative_to(BASE)}")
                    continue
                target = pack / f.relative_to(BASE)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, target)
    archive = shutil.make_archive(str(pack), "zip", pack)
    print("В архиве:", sorted(str(p.relative_to(pack)) for p in pack.rglob("*") if p.is_file()))
    files.download(archive)
else:
    print("Локальный запуск: результаты уже лежат в папках репозитория")